In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as plty
import math
import geoip2.database
from user_agents import parse

In [2]:

df = pd.read_csv("./data/int/cybersecurity_attacks.csv")

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
# Count Device Information that occured >= 30 times
df.value_counts("Device Information")[df.value_counts("Device Information") >= 30]

In [3]:
df['Browser_family'] = df['Device Information'].apply(lambda x: parse(x).browser.family if parse(x).browser.family is not None else pd.NA)
df['Browser_major'] = df['Device Information'].apply(lambda x: parse(x).browser.version[0] if parse(x).browser.version[0] is not None else pd.NA)
# df['Browser_minor'] = df['Device Information'].apply(lambda x: parse(x).browser.version[1] if parse(x).browser.version[1] is not None else pd.NA)
df['OS_family'] = df['Device Information'].apply(lambda x: parse(x).os.family if parse(x).os.family is not None else pd.NA)
df['OS_major'] = df['Device Information'].apply(
    lambda x: parse(x).os.version[0] if len(parse(x).os.version) > 0 and parse(x).os.version[0] is not None else pd.NA
)
# df['OS_minor'] = df['Device Information'].apply(
#     lambda x: parse(x).os.version[1] if len(parse(x).os.version) > 1 and parse(x).os.version[1] is not None else pd.NA
# )
# df['OS_patch'] = df['Device Information'].apply(
#     lambda x: parse(x).os.version[2] if len(parse(x).os.version) > 2 and parse(x).os.version[2] is not None else pd.NA
# )
df['Device_family'] = df['Device Information'].apply(lambda x: parse(x).device.family if parse(x).device.family is not None else pd.NA)
# df['Device_brand'] = df['Device Information'].apply(lambda x: parse(x).device.brand if parse(x).device.brand is not None else pd.NA)

In [ ]:
# df[['OS_family','OS_major', 'OS_minor', 'OS_patch', 'Browser_major', 'Browser_minor', 'Browser_family', 'Device_family', 'Device_brand']].isna().sum()
df[['OS_family','OS_major', 'Browser_major', 'Browser_family', 'Device_family']].isna().sum()

OS_family            0
OS_major          7171
Browser_major        0
Browser_family       0
Device_family        0
dtype: int64

In [9]:
# OS_major has 7171 missing values
df.drop(columns='OS_major', inplace=True)

In [ ]:
# drop feature OS_minor, OS_patch, Device_brand as it has too many missing values (25k, 29k, 23k respectively), also to reduce dimensionality
# also noise or overfitting?
# df = df.drop(columns=['OS_minor', 'OS_patch', 'Device_brand'])
# df = df.drop(columns=['Device_brand'])

In [7]:
df[['OS_family','OS_major', 'Browser_major', 'Browser_family', 'Device_family']].nunique(dropna=False)

OS_family          5
OS_major          22
Browser_major     66
Browser_family     9
Device_family      8
dtype: int64

In [ ]:
df_concatDeviceCheck = pd.DataFrame()
df_concatDeviceCheck['OS_family_major'] = df['OS_family'].astype(str) + " " + df['OS_major'].astype(str)


In [ ]:
df_concatDeviceCheck['Browser_family_major'] = df['Browser_family'].astype(str) + " " + df['Browser_major'].astype(str)
df_concatDeviceCheck['Browser_family_major_minor'] = df['Browser_family'].astype(str) + " " + df['Browser_major'].astype(str) + " " + df['Browser_minor'].astype(str)


In [ ]:
#33 combination of OS_family and OS_major
df_concatDeviceCheck['OS_family_major'].value_counts()

In [ ]:
#200 combination of Browser_family and Browser_major
df_concatDeviceCheck['Browser_family_major'].value_counts()

In [ ]:
#473 combination of Browser_family and Browser_major
df_concatDeviceCheck['Browser_family_major_minor'].value_counts()

In [ ]:
df_concatDeviceCheck['Attack Type'] = df['Attack Type']

In [ ]:
df_concatDeviceCheck['Browser_family'] = df['Browser_family']
df_concatDeviceCheck['OS_family'] = df['OS_family']
df_concatDeviceCheck['Device_family'] = df['Device_family']

In [ ]:
df_concatDeviceCheck.columns

In [ ]:
df_concatDeviceCheck.head(5)

In [ ]:
# Some OS_family_major show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['OS_family_major'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['OS_family_major'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="OS_family_major",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Some Browser_family_major_minor show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['Browser_family_major_minor'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['Browser_family_major_minor'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="Browser_family_major_minor",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Some Browser_family_major show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['Browser_family_major'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['Browser_family_major'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="Browser_family_major",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Some OS_family_major show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['OS_family'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['OS_family'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="OS_family",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Some Browser_family show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['Browser_family'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['Browser_family'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="Browser_family",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Some Device_family show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['Device_family'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['Device_family'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="Device_family",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [4]:
# ("Jamshedpur, Sikkim").split(",")[0]
df["GeoLoc_City"] = df["Geo-location Data"].apply(lambda x: x.split(",")[0])

In [5]:
df.columns

Index(['Timestamp', 'Source IP Address', 'Destination IP Address',
       'Source Port', 'Destination Port', 'Protocol', 'Packet Length',
       'Packet Type', 'Traffic Type', 'Payload Data', 'Malware Indicators',
       'Anomaly Scores', 'Alerts/Warnings', 'Attack Type', 'Attack Signature',
       'Action Taken', 'Severity Level', 'User Information',
       'Device Information', 'Network Segment', 'Geo-location Data',
       'Proxy Information', 'Firewall Logs', 'IDS/IPS Alerts', 'Log Source',
       'Browser_family', 'Browser_major', 'OS_family', 'OS_major',
       'Device_family', 'GeoLoc_City'],
      dtype='object')

In [ ]:
from user_agents import parse as ua_parse

def deviceType(ua_string):
    try:
        if not ua_string or pd.isna(ua_string):
            return pd.NA
        ua = ua_parse(ua_string)           # user_agents.parse -> object with is_mobile/is_tablet/is_pc
        if getattr(ua, "is_mobile", False):
            return "Mobile"
        if getattr(ua, "is_tablet", False):
            return "Tablet"
        if getattr(ua, "is_pc", False):
            return "PC"
        return "Unknown"
    except Exception:
        return "Unknown"

# apply
df['Device_type'] = df['Device Information'].apply(deviceType)
# deviceType("Mozilla/5.0 (iPhone; CPU iPhone OS 5_1_1 like Mac OS X) AppleWebKit/533.0 (KHTML, like Gecko) CriOS/51.0.844.0 Mobile/22B509 Safari/533.0")

In [ ]:
# no bot detected
df['is_bot'] = df['Device Information'].apply(lambda x: ua_parse(x).is_bot)
ua_parse("Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)").is_bot

In [ ]:
df_concatDeviceCheck['Device_type'] = df['Device_type']

In [ ]:
# Some Device_type show more frequently attacked however fairly distributed to Attack Type
top_n = 10
top_os = df_concatDeviceCheck['Device_type'].value_counts().head(top_n).index
df_top = df_concatDeviceCheck[df_concatDeviceCheck['Device_type'].isin(top_os)]

plt.figure(figsize=(14, 6))
sns.histplot(
    x="Device_type",
    hue="Attack Type",
    multiple="stack",
    data=df_top
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df.isna().sum()

In [ ]:
# display non-numeric columns
df.select_dtypes(exclude=[np.number]).columns.tolist()

In [ ]:
# ASK: only 1 destionation IP appears in source IPs, what does this mean? cause both looks inbound traffic.
# find destination IPs that appear in Source IPs without modifying df
destIPinSourceIP = df['Destination IP Address'].isin(df['Source IP Address'])
matched_rows = df[destIPinSourceIP].copy()  # recuperate whole rows
matched_rows

# df.drop(columns=['Dest_in_Source'])

In [ ]:
# take the first three octets of the IP as a list
split = "103.216.15.12".split(".")[:3]
ip3 = ".".join(split)
ip3

In [ ]:
# Split all IPs to /24 subnet ==> local subnet/Single host
def splitIP24(ip):
    if pd.isna(ip):
        return pd.NA
    return ".".join(ip.split(".")[:3])

# Split all IPs to /16 subnet ==> network owner/region (ISP/backbone provider)
def splitIP16(ip):
    if pd.isna(ip):
        return pd.NA
    return ".".join(ip.split(".")[:2])

df['Source IP Address_24'] = df['Source IP Address'].apply(splitIP24)
df['Source IP Address_16'] = df['Source IP Address'].apply(splitIP16)

df['Destination IP Address_24'] = df['Destination IP Address'].apply(splitIP24)
df['Destination IP Address_16'] = df['Destination IP Address'].apply(splitIP16)

df['Proxy Information_24'] = df['Proxy Information'].apply(splitIP24)
df['Proxy Information_16'] = df['Proxy Information'].apply(splitIP16)


In [ ]:
# Check private range
# • 10.0.0.0 – 10.255.255.255 
# • 172.16.0.0 – 172.31.255.255 
# • 192.168.0.0 – 192.168.255.255

def isPrivateIP(ip):
    if pd.isna(ip):
        return pd.NA
    octets = ip.split(".")
    first = int(octets[0])
    second = int(octets[1])
    if first == 10:
        return True
    elif first == 172 and 16 <= second <= 31:
        return True
    elif first == 192 and second == 168:
        return True
    else:
        return False

df['Source_IP_Private'] = df['Source IP Address'].apply(isPrivateIP)
df['Destination_IP_Private'] = df['Destination IP Address'].apply(isPrivateIP)
df['Proxy_IP_Private'] = df['Proxy Information'].apply(isPrivateIP)

In [ ]:
df['useProxy'] = df['Proxy Information'].apply(lambda x: False if pd.isna(x) else True)

In [ ]:
df['useProxy'].value_counts(dropna=False)

In [ ]:
df['Proxy_IP_Private'].value_counts(dropna=False)


In [ ]:
df['Proxy_IP_Private'] = df['Proxy_IP_Private'].apply(lambda x: 'noProxy' if pd.isna(x) else x)
df['Proxy_IP_Private'].value_counts(dropna=False)

In [ ]:
with geoip2.database.Reader('./data/ext/GeoLite2-Country_20251212/GeoLite2-Country.mmdb') as reader:
    response = reader.country(df.loc[0, 'Source IP Address'])
    print(response)
    # print(response.continent.names)

In [ ]:
df_test = df.head(2)
df_test

In [ ]:
import geoip2.errors
with geoip2.database.Reader('./data/ext/GeoLite2-Country_20251212/GeoLite2-Country.mmdb') as reader:
    def _lookup_ip(ip):
        # 1. Handle missing/NA values
        if not ip or pd.isna(ip):
            return {
                'ContinentName': pd.NA,
                'CountryName': pd.NA,
            }
        
        # 2. Perform external lookup
        try:
            response = reader.country(ip)
            
            # 3. RETURN A DICTIONARY of all desired features
            return {
                'ContinentName': response.continent.name,
                'CountryName': response.country.name,
            }
        
        except geoip2.errors.AddressNotFoundError:
            # 4. Return NA dictionary for not found IPs
            return {
                'ContinentName': pd.NA,
                'CountryName': pd.NA,
                # 'ASN': pd.NA,
                # 'Is_Private': False
            }
        except Exception:
            # 5. Return NA dictionary for other errors
            return {
                'ContinentNamee': pd.NA,
                'CountryName': pd.NA,
                # 'ASN': pd.NA,
                # 'Is_Private': False
            }

    # --- Applying the Function ---
    geo_results_sourceIP = df['Source IP Address'].apply(_lookup_ip)
    geo_results_destIP = df['Destination IP Address'].apply(_lookup_ip)
    geo_results_proxyIP = df['Proxy Information'].apply(_lookup_ip)

    # print('geo_results_sourceIP', geo_results_sourceIP)
    # Convert list of dictionaries to DataFrame
    geo_df_sourceIP = pd.DataFrame(geo_results_sourceIP.tolist(), index=geo_results_sourceIP.index)
    # print('geo_df_sourceIP', geo_df_sourceIP)
    geo_df_destIP = pd.DataFrame(geo_results_destIP.tolist(), index=geo_results_destIP.index)
    # print('geo_df_destIP', geo_df_destIP)

    geo_df_proxyIP = pd.DataFrame(geo_results_proxyIP.tolist(), index=geo_results_proxyIP.index)

    # test to df_test before real df
    # df_test.loc[:, ('ContinentName_Source', 'CountryName_Source')] = geo_df_sourceIP[['ContinentName', 'CountryName']]
    # Source IP
    df['ContinentName_Source'] = geo_df_sourceIP['ContinentName']
    df['CountryName_Source'] = geo_df_sourceIP['CountryName']
    # df.loc[:, ('ContinentName_Source', 'CountryName_Source')] = geo_df_sourceIP[['ContinentName', 'CountryName']]
    # print('am here')
    
    # Destination IP
    df['ContinentName_Destination'] = geo_df_destIP['ContinentName']
    df['CountryName_Destination'] = geo_df_destIP['CountryName']

    # Proxy IP
    df['ContinentName_Proxy'] = geo_df_proxyIP['ContinentName']
    df['CountryName_Proxy'] = geo_df_proxyIP['CountryName']


In [ ]:
with geoip2.database.Reader('./data/ext/GeoLite2-City_20251212/GeoLite2-City.mmdb') as reader:
    response = reader.city(df.loc[0, 'Source IP Address'])
    print(response)
    print(response.city.name)
    print(response.location.latitude)
    print(response.location.longitude)
    # print(response.postal.code)

In [ ]:
df.columns

In [ ]:
grouped = df.groupby('Source_IP_Private')['Attack Type'].value_counts()
percentages = (grouped / grouped.groupby(level=0).sum()) * 100
print(percentages)
print(grouped)

In [ ]:
grouped = df.groupby('useProxy')['Attack Type'].value_counts()
percentages = (grouped / grouped.groupby(level=0).sum()) * 100
print(percentages)
print(grouped)

In [ ]:
# 8333 unique same source ports appear more than once
# of total 29761 source ports, means around 30%
# Maybe can't use this cause when we divide the dataset in train/test, we won't have same result of frequency of source port appearing
df['Source Port'].value_counts()[df['Source Port'].value_counts() > 1]

In [ ]:
# 8217 unique destination ports appear more than once
# of total 29895 unique destination ports, means around 27.5%
# Maybe can't use this cause when we divide the dataset in train/test, we won't have same result of frequency of destination port appearing
df['Destination Port'].value_counts()[df['Destination Port'].value_counts() > 1]

In [ ]:
# df['Destination Port'].nunique() # 29895 unique destination ports
df['Source Port'].nunique() # 29761 unique source ports

In [ ]:
# Treat ports
# HTTPS 443, SSH 22, DNS 53, FTP 20-21) 
# Well-known (0–1023) → common services (HTTP, FTP, DNS, etc.) 
# User/Registered Ports (1024-49151) → used by Server, Application, Database, VPN (HTTP alternate 8080, MySQL 3306) 
# Dynamic and/or Private Ports (49152-65535) → temporary/ephemeral ports for outgoing connections, used by Clients. Rapid rotation may signal botnets. 
# Plot to see distribution of these different ports vs packet length 
# Also vs protocol/traffic type/attack type, etc ? Any meaning? 
def portCategory(port):
    if pd.isna(port):
        return pd.NA
    elif 0 <= port <= 1023:
        return 'Wellknown'
    elif 1024 <= port <= 49151:
        return 'Registered'
    elif 49152 <= port <= 65535:
        return 'Dynamic'
    else:
        return 'Invalid'

df['Source Port_Cat'] = df['Source Port'].apply(portCategory)
df['Destination Port_Cat'] = df['Destination Port'].apply(portCategory)

In [ ]:
df.groupby('Source Port')['Attack Type'].value_counts().sort_values(ascending=False)

In [ ]:
ct = pd.crosstab(df['Source Port'], [df['Source Port_Cat'], df['Attack Type']])
ct['Total'] = ct.sum(axis=1)
ct = ct.sort_values('Total', ascending=False).drop('Total', axis=1)
ct

In [ ]:
ct = pd.crosstab(df['Source Port_Cat'], [df['Attack Type']])
ct['Total'] = ct.sum(axis=1)
ct = ct.sort_values('Total', ascending=False).drop('Total', axis=1)
ct

In [ ]:
df['Source Port'].value_counts().sort_values(ascending=False)

In [ ]:
df['Source Port_Cat'].value_counts().sort_values(ascending=False)

In [ ]:
df['Destination Port_Cat'].value_counts().sort_values(ascending=False)

In [ ]:
ct = pd.crosstab(df['Destination Port_Cat'], [df['Attack Type'], df['Severity Level']])
ct['Total'] = ct.sum(axis=1)
ct = ct.sort_values('Total', ascending=False).drop('Total', axis=1)
ct

In [ ]:
sns.displot(x='Source Port_Cat', hue='Attack Type', multiple='stack', col='Severity Level', data=df)
plt.show()

In [ ]:
sns.displot(x='Source Port_Cat', hue='Attack Type', multiple='stack', data=df)
plt.show()

In [ ]:
sns.displot(x='Destination Port_Cat', hue='Attack Type', multiple='stack', data=df)
plt.show()

In [ ]:
import geoip2.errors
with geoip2.database.Reader('./data/ext/GeoLite2-City_20251212/GeoLite2-City.mmdb') as reader:
    def _lookup_ip(ip):
        # 1. Handle missing/NA values
        if not ip or pd.isna(ip):
            return {
                'CityName': pd.NA,
                # 'CityLat_Source': pd.NA,
                # 'CityLon': pd.NA,
            }
        
        # 2. Perform external lookup
        try:
            response = reader.city(ip)
            
            # 3. RETURN A DICTIONARY of all desired features
            return {
                    'CityName': response.city.name,
                    # 'CityLat': response.location.latitude,
                    # 'CityLon': response.location.longitude,
            }
        
        except geoip2.errors.AddressNotFoundError:
            # 4. Return NA dictionary for not found IPs
            return {
                'CityName': pd.NA,
                # 'CityLat': pd.NA,
                # 'CityLon': pd.NA,
                # 'ASN': pd.NA,
                # 'Is_Private': False
            }
        except Exception:
            # 5. Return NA dictionary for other errors
            return {
                'CityName': pd.NA,
                # 'CityLat': pd.NA,
                # 'CityLon': pd.NA,
                # 'ASN': pd.NA,
                # 'Is_Private': False
            }

    # --- Applying the Function ---
    geo_results_sourceIP = df['Source IP Address'].apply(_lookup_ip)
    geo_results_destIP = df['Destination IP Address'].apply(_lookup_ip)
    geo_results_proxyIP = df['Proxy Information'].apply(_lookup_ip)

    # print(geo_results_sourceIP)
    # Convert list of dictionaries to DataFrame
    geo_df_sourceIP = pd.DataFrame(geo_results_sourceIP.tolist(), index=geo_results_sourceIP.index)
    geo_df_destIP = pd.DataFrame(geo_results_destIP.tolist(), index=geo_results_destIP.index)
    geo_df_proxyIP = pd.DataFrame(geo_results_proxyIP.tolist(), index=geo_results_proxyIP.index)
    # print(geo_df_sourceIP)

    # test to df_test before real df
    # df_test.loc[:, ('CityName_Source', 'CityLat_Source', 'CityLon_Source')] = geo_df[['CityName_Source', 'CityLat_Source', 'CityLon_Source']]
    df['CityName_Proxy'] = geo_df_proxyIP['CityName']
    # df['CityLat_Proxy'] = geo_df_proxyIP['CityLat']
    # df['CityLon_Proxy'] = geo_df_proxyIP['CityLon']

    df['CityName_Destination'] = geo_df_destIP['CityName']
    # df['CityLat_Destination'] = geo_df_destIP['CityLat']
    # df['CityLon_Destination'] = geo_df_destIP['CityLon']

    df['CityName_Source'] = geo_df_sourceIP['CityName']
    # df['CityLat_Source'] = geo_df_sourceIP['CityLat']
    # df['CityLon_Source'] = geo_df_sourceIP['CityLon']

    # why codes below didnt work??
    # df.loc[:, ('CityName_Source', 'CityLat_Source', 'CityLon_Source')] = geo_df_sourceIP[['CityName', 'CityLat', 'CityLon']]
    # df.loc[:, ('CityName_Destination', 'CityLat_Destination', 'CityLon_Destination')] = geo_df_destIP[['CityName', 'CityLat', 'CityLon']]
    # df.loc[:, ('CityName_Proxy', 'CityLat_Proxy', 'CityLon_Proxy')] = geo_df_proxyIP[['CityName', 'CityLat', 'CityLon']]

In [ ]:
# How to treat timestamps?
# Timeseries?
df['Year'] = df['Timestamp'].apply(lambda x: pd.to_datetime(x).year)
df['Month'] = df['Timestamp'].apply(lambda x: pd.to_datetime(x).month)
df['Day'] = df['Timestamp'].apply(lambda x: pd.to_datetime(x).year)
# https://www.indialawoffices.com/legal-articles/work-hours-and-office-timing-in-india
# Daily working hours range from 8-10 hours.
# 8 to 7 (pause 1h) - 10h (assumption)
df['Working_hours'] = pd.to_datetime(df['Timestamp']).apply(lambda x: 'Yes' if 7 <= x.hour <= 19 else "No")
# pd.to_datetime(df.loc[6, 'Timestamp']).year

In [ ]:
df.isna().sum()
#CityName_Source 21401
#CityName_Destination 21568
#CityName_Proxy 30751

In [ ]:
df['Malware Indicators'].unique() # Only has 2 values: IoC Detected (1), nan (0)
df['Malware Indicators'] = df['Malware Indicators'].apply(lambda x: 0 if pd.isna(x) else 1)

In [ ]:
df['Alerts/Warnings'].unique() # # Only has 2 values: Alert Triggered (1), nan (0)
df['Alerts/Warnings'] = df['Alerts/Warnings'].apply(lambda x: 0 if pd.isna(x) else 1)


In [ ]:
df.drop(columns=['CityName_Source','CityName_Destination','CityName_Proxy'], inplace=True)

In [ ]:
# df_corr = df_corr.drop(columns=['Device Information', 'Timestamp', 'Geo-location Data', 'Source IP Address', 'Destination IP Address', 'Proxy Information'])
# df_corr = df_corr.drop(columns=['ContinentName_Source', 'ContinentName_Destination', 'ContinentName_Proxy']) 
                                # 'CityLat_Proxy', 'CityLon_Proxy', 'CityLat_Destination', 'CityLon_Destination', 'CityLon_Source', 'CityLat_Source'])
# df_corr = df_corr.drop(columns=['OS_major', 'Proxy Information_16', 'Proxy Information_24'])
# df_corr = df_corr.drop(columns=['Payload Data', 'User Information', 'Destination IP Address_24', 'Source IP Address_24'])
# df_corr = df_corr.drop(columns=['CityName_Proxy', 'CityName_Source', 'CityName_Destination'])

In [ ]:
print((df_corr['is_Proxy'] == False).sum())
print((df_corr['is_Proxy'] == True).sum())

In [ ]:
df_corr['Proxy_IP_Private'].fillna('no IP', inplace=True)
df_corr['CountryName_Source'].fillna('not identified', inplace=True)
df_corr['CountryName_Destination'].fillna('not identified', inplace=True)
# df_corr['CityName_Source'].fillna('not identified', inplace=True)
# df_corr['CityName_Destination'].fillna('not identified', inplace=True)
# df_corr.loc[df_corr['is_Proxy'] == False, ['CountryName_Proxy', 'CityName_Proxy']] = \
#     df_corr.loc[df_corr['is_Proxy'] == False, ['CountryName_Proxy', 'CityName_Proxy']].fillna('no_Proxy')
# df_corr.loc[df_corr['is_Proxy'] == True, ['CountryName_Proxy', 'CityName_Proxy']] = \
#     df_corr.loc[df_corr['is_Proxy'] == True, ['CountryName_Proxy', 'CityName_Proxy']].fillna('not_identified')

In [ ]:
df_corr['Alerts/Warnings'] = df_corr['Alerts/Warnings'].map({'Alert Triggered': 1, 'Alert Triggered-n/a': 0})
df_corr['IDS/IPS Alerts'] = df_corr['IDS/IPS Alerts'].map({'Alert Data': 1, 'Alert Data-n/a': 0})
df_corr['Firewall Logs'] = df_corr['Firewall Logs'].map({'Log Data': 1, 'Log Data-n/a': 0})
df_corr['Working_hours'] = df_corr['Working_hours'].map({'Yes': 1, 'No': 0})
df_corr['is_Proxy'] = df_corr['is_Proxy'].map({'True': 1, 'False': 0})
df_corr['Malware Indicators'] = df_corr['Malware Indicators'].map({'IoC Detected': 1, 'IoC Detected-n/a': 0})
df_corr['Attack Signature'] = df_corr['Attack Signature'].map({'Known Pattern A': 1, 'Known Pattern B': 0})

In [ ]:
df_corr['isProxy'] = df_corr['Proxy_IP_Private'].map({'False': 1, 'True': 1, 'no IP': 0})
# df_corr['isProxy'] = df_corr['Proxy IP_Private'].astype(str).map({'False': 1, 'True': 1, 'no IP': 0})

In [ ]:
df_corr.columns

In [ ]:
df_corr

In [ ]:
df_corr['CountryName_Source'].str.contains('India').sum()

In [ ]:
df_corr['CountryName_Destination'].str.contains('India').sum()

In [ ]:
df_corr['CountryName_Proxy'].str.contains('India').sum()

In [ ]:
(df_corr['CountryName_Source'].str.contains('India', na=False) & \
 df_corr['CountryName_Destination'].str.contains('India', na=False)).sum()

In [ ]:
(df_corr['CountryName_Source'].str.contains('India', na=False) & \
 df_corr['CountryName_Proxy'].str.contains('India', na=False)).sum()

In [ ]:
(df_corr['CountryName_Destination'].str.contains('India', na=False) & \
 df_corr['CountryName_Proxy'].str.contains('India', na=False)).sum()

In [ ]:
(df_corr['CountryName_Destination'].str.contains('India', na=False) & \
 df_corr['CountryName_Source'].str.contains('India', na=False) &\
 df_corr['CountryName_Proxy'].str.contains('India', na=False)).sum()

In [ ]:
df_corr['Malware Indicators'] = df_corr['Malware Indicators'].fillna('IoC Detected-n/a')
df_corr['Alerts/Warnings'] = df_corr['Alerts/Warnings'].fillna('Alert Triggered-n/a')
df_corr['IDS/IPS Alerts'] = df_corr['IDS/IPS Alerts'].fillna('Alert Data-n/a')
df_corr['Firewall Logs'] = df_corr['Firewall Logs'].fillna('Log Data-n/a')

In [ ]:
df_corr.head(5)

In [ ]:
df_corr.head(5)

In [ ]:
df_corr.columns

In [ ]:
grouped = df.groupby('Destination_IP_Private')['Attack Type'].value_counts()
percentages = (grouped / grouped.groupby(level=0).sum()) * 100
print(percentages)
print(grouped)


In [ ]:
# test to df_test before real df
# df_test
df.head(5)

In [ ]:
# Treat ASN for IP source?

In [ ]:
df.head(5)

In [ ]:
filter = df['Protocol'] == 'ICMP'
ct = df.loc[filter, ['Traffic Type', 'Attack Type']].value_counts().unstack(fill_value=None)
ct

In [ ]:
filter = df['Protocol'] == 'ICMP'
ct = pd.crosstab(df.loc[filter, 'Traffic Type'], df.loc[filter, 'Attack Type'])
ct

In [ ]:
df.head(5)

In [ ]:
# Plot Browser_family vs Attack Type (stacked bar for top N browser families)
# create a contingency table: rows = Browser_family, columns = Attack Type
# Plot relation between Browser_family, Attack Type, and Action Taken (stacked bars by Action Taken per Browser, one subplot per Attack Type)
top_n = 10
top_browsers = df['Browser_family'].value_counts().head(top_n).index
df_top = df[df['Browser_family'].isin(top_browsers)].copy()

# cross-tab: (Browser_family, Attack Type) x Action Taken
ct3 = pd.crosstab([df_top['Browser_family'], df_top['Attack Type']], df_top['Action Taken'])

attack_types = df_top['Attack Type'].unique()
ncols = len(attack_types)
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 6), sharey=True)

if ncols == 1:
    axes = [axes]

for ax, atk in zip(axes, attack_types):
    try:
        # select rows for this attack type and reset index to have Browser_family as index
        df_plot = ct3.xs(atk, level='Attack Type')
    except KeyError:
        # no rows for this attack type in top browsers
        df_plot = pd.DataFrame()
    if not df_plot.empty:
        df_plot.plot(kind='bar', stacked=True, ax=ax, legend=False)
    ax.set_title(f'Action Taken by Browser (Attack: {atk})')
    ax.set_xlabel('Browser family')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

# common legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Action Taken', bbox_to_anchor=(1.02, 0.9))
plt.tight_layout(rect=[0,0,0.85,1])
plt.show()

In [ ]:
# Plot OS_family vs Attack Type (stacked bar for top N browser families)
# create a contingency table: rows = Browser_family, columns = Attack Type
ct = pd.crosstab(df['OS_family'], df['Attack Type'])

# show full table (or limit to top N browser families by total count)
top_n = 10
top_browsers = ct.sum(axis=1).sort_values(ascending=False).head(top_n).index
ct_top = ct.loc[top_browsers]

# display table
ct_top

# stacked bar plot for the top N browser families
ct_top.plot(kind='bar', stacked=True, figsize=(12,6))
plt.xlabel('OS family')
plt.ylabel('Count')
plt.title(f'OS family vs Attack Type (top {top_n} OS families)')
plt.legend(title='Attack Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Plot Source Port_Cat vs Attack Type (stacked bar for top N browser families)
# create a contingency table: rows = Browser_family, columns = Attack Type
ct = pd.crosstab(df['Source Port_Cat'], df['Attack Type'])

# show full table (or limit to top N browser families by total count)
# top_n = 10
top_browsers = ct.sum(axis=1).sort_values(ascending=False).head(top_n).index
ct_top = ct.loc[top_browsers]

# display table
ct_top

# stacked bar plot for the top N browser families
ct_top.plot(kind='bar', stacked=True, figsize=(12,6))
plt.xlabel('Source Port_Cat')
plt.ylabel('Count')
plt.title(f'Source Port_Cat vs Attack Type (top {top_n} Source Port_Cat)')
plt.legend(title='Attack Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('Attack Type')['Anomaly Scores'].describe()

In [ ]:
df.groupby('Severity Level')['Anomaly Scores'].describe()

In [ ]:
df.groupby(['Attack Type', 'Severity Level'])['Anomaly Scores'].describe()

In [ ]:
# Fill ICMP_anomaly column: better? or no need cause anyway ICMP can be detected by Protocol feature
df['ICMP_anomaly'] = df.apply(
    lambda x: 1 if (
        x['Protocol'] == "ICMP" and (
            x['Traffic Type'] == "HTTP" or 
            x['Traffic Type'] == "DNS" or 
            x['Traffic Type'] == "FTP"
        )
    ) else 0, 
    axis=1
)

In [ ]:
df.groupby(['ICMP_anomaly', 'Traffic Type'])['Attack Type'].value_counts()

In [ ]:
df

In [ ]:
df.corr(numeric_only=True)

In [ ]:
cols_to_check = [
    "Attack Signature",
    "Severity Level",
    "Action Taken",
    "IDS/IPS Alerts",
    "Malware Indicators",
    "Alerts/Warnings",
    "Network Segment",
    "Firewall Logs",
    "Log Source"
]
for col in cols_to_check:
    print(f"\n--- {col} ---")
    print(df.groupby('Attack Type')[col].value_counts(dropna=False))

In [ ]:
# is it good news cause our model won't have bias for this colums cause it has pretty  fair distribution? or there is other meaning?
df.groupby('Attack Type')['Firewall Logs'].value_counts()

In [ ]:
df.groupby('Attack Type')['Firewall Logs'].value_counts(dropna=False)

In [ ]:
# is it good news cause our model won't have bias for this colums cause it has pretty  fair distribution? or there is other meaning?
df.groupby('Attack Type')['Alerts/Warnings'].value_counts()

In [ ]:
df.groupby('Attack Type')['Alerts/Warnings'].value_counts(dropna=False)

In [ ]:
# is it good news cause our model won't have bias for this colums cause it has pretty  fair distribution? or there is other meaning?
df.groupby('Attack Type')['Malware Indicators'].value_counts()

In [ ]:
df.groupby('Attack Type')['Malware Indicators'].value_counts(dropna=False)

In [ ]:
df.groupby('Attack Type')['IDS/IPS Alerts'].value_counts()

In [ ]:
df.groupby('Attack Type')['IDS/IPS Alerts'].value_counts(dropna=False)

In [ ]:
sns.histplot(x="Malware Indicators",hue="Attack Type",multiple="stack", data=df)
plt.show()

In [ ]:
df.groupby('Attack Type')['Firewall Logs'].value_counts()

In [ ]:
# entropy doesn't make any sense as the payload data seems like synthetic (need to check with Hanna).
from collections import Counter

def entropy_bytes(s):
    b = str(s).encode('utf-8', errors='ignore')
    if not b: return 0.0
    counts = Counter(b)
    total = len(b)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

df['Entropy'] = df['Payload Data'].apply(entropy_bytes)
df['Band'] = df['Entropy'].apply(lambda e: 'High' if e > 7 else ('Medium' if e >= 5 else 'Low'))
print(df['Band'].value_counts())

In [ ]:
df.groupby('Attack Type')['Packet Length'].describe()

In [ ]:
# doesn't show spike of Blocked if Severity Level is High
df.groupby(['Attack Type', 'Severity Level'])['Action Taken'].value_counts()

In [ ]:
# Doesn't show that IDS/IPS Alert trigger Alerts/Warnings
df.groupby('IDS/IPS Alerts')['Alerts/Warnings'].value_counts(dropna=False)

In [ ]:
# Doesn't show that certain signature is assigned to attack type
df.groupby('Attack Type')['Attack Signature'].value_counts(dropna=False)